# 10 — Association

## Objectives
- Understand association relationships
- Distinguish: Association vs Aggregation vs Composition
- Implement bidirectional and unidirectional associations

## Association Types
```
Association  → A uses B (general 'uses' relationship)
Aggregation  → A has B (weak ownership, B can exist alone)
Composition  → A has B (strong ownership, B dies with A)
```

In [3]:
// Unidirectional association: Student uses Library
class Library {
    private String name;
    private java.util.Map<String, Integer> catalog = new java.util.HashMap<>();
    
    Library(String name) { this.name = name; }
    
    void addBook(String isbn, int copies) { catalog.put(isbn, copies); }
    
    boolean checkout(String isbn) {
        int copies = catalog.getOrDefault(isbn, 0);
        if (copies <= 0) { System.out.println("Book " + isbn + " not available"); return false; }
        catalog.put(isbn, copies - 1);
        System.out.println("Book " + isbn + " checked out. Remaining: " + (copies-1));
        return true;
    }
    
    String getName() { return name; }
}

class Student {
    private String name;
    private Library library; // Student USES Library (association)
    
    Student(String name, Library library) {
        this.name = name;
        this.library = library; // passed in, not created here
    }
    
    void borrowBook(String isbn) {
        System.out.print(name + " borrowing from " + library.getName() + ": ");
        if (library.checkout(isbn)) System.out.println("SUCCESS");
    }
}

Library lib = new Library("Central Library");
lib.addBook("ISBN-001", 3);
lib.addBook("ISBN-002", 1);

// Multiple students using same library
Student alice = new Student("Alice", lib);
Student bob = new Student("Bob", lib);

alice.borrowBook("ISBN-001");
bob.borrowBook("ISBN-001");
alice.borrowBook("ISBN-002");
bob.borrowBook("ISBN-002"); // Should fail - last copy taken

Alice borrowing from Central Library: Book ISBN-001 checked out. Remaining: 2
SUCCESS
Bob borrowing from Central Library: Book ISBN-001 checked out. Remaining: 1
SUCCESS
Alice borrowing from Central Library: Book ISBN-002 checked out. Remaining: 0
SUCCESS
Bob borrowing from Central Library: Book ISBN-002 not available


## Mini Challenge
Model: `Doctor` treats many `Patient` objects. `Patient` can be treated by many `Doctor` objects. Implement this M:N association.

In [6]:
import java.util.HashSet;
import java.util.Set;

public class Patient {
    private String name;
    private Set<Doctor> doctors;

    public Patient(String name) {
        this.name = name;
        this.doctors = new HashSet<>();
    }

    public String getName() { return name; }
    public Set<Doctor> getDoctors() { return doctors; }

    // Helper method to add a doctor
    public void addDoctor(Doctor doctor) {
        if (!this.doctors.contains(doctor)) {
            this.doctors.add(doctor);
            doctor.addPatient(this); // Keeps the relationship bidirectional
        }
    }

    @Override
    public String toString() {
        return "Patient: " + name;
    }
}

In [5]:
import java.util.HashSet;
import java.util.Set;
import java.util.stream.Collectors;

public class Doctor {
    private String name;
    private Set<Patient> patients;

    public Doctor(String name) {
        this.name = name;
        this.patients = new HashSet<>();
    }

    public String getName() { return name; }
    public Set<Patient> getPatients() { return patients; }

    // Helper method to add a patient
    public void addPatient(Patient patient) {
        if (!this.patients.contains(patient)) {
            this.patients.add(patient);
            patient.addDoctor(this); // Keeps the relationship bidirectional
        }
    }

    public void displayPatients() {
        String patientNames = patients.stream()
            .map(Patient::getName)
            .collect(Collectors.joining(", "));
        System.out.println("Dr. " + name + " is treating: [" + patientNames + "]");
    }

    @Override
    public String toString() {
        return "Doctor: " + name;
    }
}

In [7]:
// Create Doctors
Doctor docHouse = new Doctor("House");
Doctor docCuddy = new Doctor("Cuddy");

// Create Patients
Patient john = new Patient("John Doe");
Patient jane = new Patient("Jane Smith");

// Establish Many-to-Many Connections
docHouse.addPatient(john); // House treats John
docHouse.addPatient(jane); // House treats Jane
docCuddy.addPatient(john); // Cuddy also treats John (Many Doctors to One Patient)

// Display Results
docHouse.displayPatients();
docCuddy.displayPatients();

// Verify reciprocity from the patient's side
String johnsDocs = john.getDoctors().stream()
    .map(Doctor::getName)
    .collect(Collectors.joining(", "));
System.out.println("John Doe's doctors: [" + johnsDocs + "]");

Dr. House is treating: [Jane Smith, John Doe]
Dr. Cuddy is treating: [John Doe]
John Doe's doctors: [Cuddy, House]
